Run `python -m src.pipeline` first, it creates `data/output/tech_news.duckdb`.

In [ ]:
from pathlib import Path
import duckdb

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

con = duckdb.connect(str(ROOT / "data/output/tech_news.duckdb"), read_only=True)

In [ ]:
con.sql("SHOW TABLES").df()

In [ ]:
con.sql("SELECT * FROM dim_company").df()

In [ ]:
# Rows per table
con.sql("""
    SELECT 'dim_company' AS table_name, COUNT(*) AS row_count FROM dim_company
    UNION ALL SELECT 'fact_article', COUNT(*) FROM fact_article
    UNION ALL SELECT 'fact_arr_observation', COUNT(*) FROM fact_arr_observation
""").df()

In [ ]:
# Exploration of articles by company
con.sql("""
    SELECT com.company_name,
        COUNT(*) as article_count,
        MIN(obs.as_of_date) as first_article_date,
        MAX(obs.as_of_date) as last_article_date,
        MIN(obs.arr_usd) as min_arr_usd,
        MAX(obs.arr_usd) as max_arr_usd
    FROM fact_arr_observation obs
    INNER JOIN dim_company com ON com.company_id = obs.company_id
    GROUP BY com.company_name
    ORDER BY article_count DESC
""").df()

In [ ]:
# Latest ARR of each company
con.sql("""
    SELECT * FROM fact_arr_observation
    WHERE company_id = ?
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY company_id
        ORDER BY as_of_date DESC, article_id DESC
    ) = 1
""", params=["openai"]).df()

In [ ]:
# Last Quarter ARR by company and year
con.sql("""
    SELECT *, YEAR(as_of_date) AS year, QUARTER(as_of_date) AS quarter
    FROM fact_arr_observation
    WHERE company_id = ?
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY company_id, YEAR(as_of_date), QUARTER(as_of_date)
        ORDER BY as_of_date DESC, article_id DESC
    ) = 1    
    ORDER BY year, quarter
""", params=["openai"]).df()

In [ ]:
# Status of the extracted ARR by article
con.sql("""
    SELECT COALESCE(arr_exclusion_reason, '(valid ARR)') AS reason, COUNT(*) AS articles
    FROM fact_article
    GROUP BY reason
    ORDER BY articles DESC
""").df()

In [ ]:
# Companies that are not in company_metadata.json (no enriched data: industry, founded year, etc...)
con.sql("""
    SELECT com.company_id, com.company_name, COUNT(art.article_id) AS articles
    FROM dim_company com
    LEFT JOIN fact_article art ON art.company_id = com.company_id
    WHERE com.in_metadata = False
    GROUP BY com.company_id, com.company_name
    ORDER BY articles DESC
""").df()